In [9]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

In [10]:
def tokens(corpus):
  arr =[]
  for i in range(len(corpus)):
    text = corpus[i]
    tokens = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
       outputs = model(**tokens)

    # Extract embeddings from the last layer
    embeddings = outputs.last_hidden_state
    # Example of mean pooling
    pooled_embeddings = torch.mean(embeddings, dim=1)
    arr.append(pooled_embeddings.numpy())
  return arr

In [11]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('vinai/bertweet-base')
model = BertModel.from_pretrained('vinai/bertweet-base')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertweetTokenizer'. 
The class this function is called from is 'BertTokenizer'.
You are using a model of type roberta to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
Some weights of BertModel were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.7.attention.self.value.bias', 'encoder.layer.3.attention.self.value.bias', 'encoder.layer.4.attention.self.key.bias', 'encoder.layer.4.attention.self.value.bias', 'encoder.layer.6.intermediate.dense.weight', 'encoder.layer.5.attention.self.value.bias', 'encoder.layer.7.attention.output.dense.bias', 'encoder.layer.6.attention.output.LayerNorm.weight', 'encoder.layer.5.interme

In [12]:
def tokens(corpus):
  arr =[]
  count = 0
  for i in range(len(corpus)):
    text = corpus[i]
    tokens = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
       outputs = model(**tokens)

    # Extract embeddings from the last layer
    embeddings = outputs.last_hidden_state
    # Example of mean pooling
    pooled_embeddings = torch.mean(embeddings, dim=1)
    count = count +1
    print(count)
    arr.append(pooled_embeddings.numpy())
  return arr



In [13]:
import pandas as pd
df = pd.read_excel('behaviour_simulation_test_company.xlsx')

In [14]:
text = df['content']

In [15]:
from transformers import BertTokenizer, BertModel
import torch
import pandas as pd

import numpy as np
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained BERT tokenizer and model on the selected device
tokenizer1 = BertTokenizer.from_pretrained('vinai/bertweet-base')
model = BertModel.from_pretrained('vinai/bertweet-base').to(device)

# Function to tokenize and get embeddings for a batch of texts
def process_batch(texts):
    # Tokenize batched texts
    tokens = tokenizer1(texts, return_tensors='pt', padding=True, truncation=True, max_length=128, return_attention_mask=False).to(device)
    with torch.no_grad():
        outputs = model(**tokens)

    # Extract embeddings from the last layer
    embeddings = outputs.last_hidden_state
    # Example of mean pooling
    pooled_embeddings = torch.mean(embeddings, dim=1)
    return pooled_embeddings.cpu().numpy()

# Process texts in batches
def process_texts_in_batches(texts, batch_size=32):
    embeddings_list = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        embeddings = process_batch(batch_texts)
        embeddings_list.append(embeddings)
    return embeddings_list

# Read data
df = pd.read_excel('behaviour_simulation_test_company.xlsx')
text = df['content']

# Process a subset of the first 1000 texts
num_texts = 10000
text_subset = text.head(num_texts).tolist()  # Convert to list

# Process texts in batches
text_enc = process_texts_in_batches(text_subset)

# Combine embeddings from batches into a single array
text_enc_combined = np.concatenate(text_enc, axis=0)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertweetTokenizer'. 
The class this function is called from is 'BertTokenizer'.
You are using a model of type roberta to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
Some weights of BertModel were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.7.attention.self.value.bias', 'encoder.layer.3.attention.self.value.bias', 'encoder.layer.4.attention.self.key.bias', 'encoder.layer.4.attention.self.value.bias', 'encoder.layer.6.intermediate.dense.weight', 'encoder.layer.5.attention.self.value.bias', 'encoder.layer.7.attention.output.dense.bias', 'encoder.layer.6.attention.output.LayerNorm.weight', 'encoder.layer.5.interme

In [16]:
text_enc_combined.shape

(10000, 768)

In [17]:
import numpy as np
from sklearn.decomposition import PCA


pca = PCA(n_components = 32)


In [18]:
text_enc_combined = pca.fit_transform(text_enc_combined)

In [19]:
import pandas as pd

# Assuming 'transformed_arr' is your transformed array (300000, 32)

# Convert the transformed array into a DataFrame
transformed_df = pd.DataFrame(text_enc_combined)

# Define the file path where you want to save the CSV file
output_file_path = 'test_text_enc_red.csv'

# Save the DataFrame to a CSV file
transformed_df.to_csv(output_file_path, index=False)

In [20]:
df

,id,date,content,username,media,inferred company
0,1,2019-01-08 18:03:10,the Heathrow drone right now <hyperlink>,bbcthree,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc
1,2,2018-01-29 10:51:17,A deal at £60m?\n\nThe latest on Pierre-Emeric...,BBCSport,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc
2,3,2019-09-05 07:25:14,"""Dealing with Boris Johnson is like someone de...",BBCPolitics,[Video(thumbnailUrl='https://pbs.twimg.com/med...,bbc
3,4,2018-03-28 18:15:00,Doesn't unlimited food &amp; drinks all night ...,BrdgstoneArena,[Photo(previewUrl='https://pbs.twimg.com/media...,bridgestone
4,5,2019-09-11 21:15:44,Apple Watch Series 5.\n\nYou’ve never seen a w...,BestBuy,[Video(thumbnailUrl='https://pbs.twimg.com/ext...,best buy
...,...,...,...,...,...,...
9995,9996,2020-01-25 01:45:07,"""So what do all these defenses mean? What they...",HouseIntelDems,[Video(thumbnailUrl='https://pbs.twimg.com/med...,house
9996,9997,2019-06-26 12:51:27,"In this week’s #MacroMemo, we explore last wee...",rbcgamnews,[Photo(previewUrl='https://pbs.twimg.com/media...,rbc
9997,9998,2019-05-06 10:37:03,When it's bank holiday Monday and someone ment...,bbceastenders,[Gif(thumbnailUrl='https://pbs.twimg.com/tweet...,bbc
9998,9999,2019-05-19 12:00:51,England boss Gareth Southgate says he is conce...,BBCSport,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc


In [21]:
a = np.zeros((10000,32))

In [22]:
a = pd.DataFrame(a)

In [23]:
df = pd.concat([df,a],axis=1 )

In [24]:
df_n = pd.read_excel('behaviour_simulation_test_company.xlsx')

In [25]:
df_1 = df.head(1000)

In [26]:
df_n

,id,date,content,username,media,inferred company
0,1,2019-01-08 18:03:10,the Heathrow drone right now <hyperlink>,bbcthree,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc
1,2,2018-01-29 10:51:17,A deal at £60m?\n\nThe latest on Pierre-Emeric...,BBCSport,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc
2,3,2019-09-05 07:25:14,"""Dealing with Boris Johnson is like someone de...",BBCPolitics,[Video(thumbnailUrl='https://pbs.twimg.com/med...,bbc
3,4,2018-03-28 18:15:00,Doesn't unlimited food &amp; drinks all night ...,BrdgstoneArena,[Photo(previewUrl='https://pbs.twimg.com/media...,bridgestone
4,5,2019-09-11 21:15:44,Apple Watch Series 5.\n\nYou’ve never seen a w...,BestBuy,[Video(thumbnailUrl='https://pbs.twimg.com/ext...,best buy
...,...,...,...,...,...,...
9995,9996,2020-01-25 01:45:07,"""So what do all these defenses mean? What they...",HouseIntelDems,[Video(thumbnailUrl='https://pbs.twimg.com/med...,house
9996,9997,2019-06-26 12:51:27,"In this week’s #MacroMemo, we explore last wee...",rbcgamnews,[Photo(previewUrl='https://pbs.twimg.com/media...,rbc
9997,9998,2019-05-06 10:37:03,When it's bank holiday Monday and someone ment...,bbceastenders,[Gif(thumbnailUrl='https://pbs.twimg.com/tweet...,bbc
9998,9999,2019-05-19 12:00:51,England boss Gareth Southgate says he is conce...,BBCSport,[Photo(previewUrl='https://pbs.twimg.com/media...,bbc


In [27]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')


In [28]:
def tokens(corpus):
  arr =[]
  for i in range(len(corpus)):
    text = corpus[i]
    tokens = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
       outputs = model(**tokens)

    # Extract embeddings from the last layer
    embeddings = outputs.last_hidden_state
    # Example of mean pooling
    pooled_embeddings = torch.mean(embeddings, dim=1)
    arr.append(pooled_embeddings.numpy())
  return arr



In [29]:
corpus = df['inferred company']

company_enc = tokens(corpus)

In [30]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def tokens(corpus):
    arr = []
    batch_size = 8  # Adjust batch size according to your resources
    for i in range(0, len(corpus), batch_size):
        texts = corpus[i:i + batch_size]
        tokens = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=128)
        tokens.to(device)
        
        with torch.no_grad():
            outputs = model(**tokens)
            embeddings = outputs.last_hidden_state
            pooled_embeddings = torch.mean(embeddings, dim=1)
            arr.extend(pooled_embeddings.cpu().numpy())
    
    return arr

# Assuming df['inferred company'] contains your corpus
corpus = df['inferred company'].tolist()

company_enc = tokens(corpus)


In [31]:
np.array(company_enc).shape

(10000, 768)

In [32]:
import numpy as np
from sklearn.decomposition import PCA


pca = PCA(n_components = 32)


In [33]:
company_enc = pca.fit_transform(company_enc)

In [34]:
import pandas as pd

# Assuming 'transformed_arr' is your transformed array (300000, 32)

# Convert the transformed array into a DataFrame
transformed_df = pd.DataFrame(company_enc)

# Define the file path where you want to save the CSV file
output_file_path = 'test_company_enc_red_comp.csv'

# Save the DataFrame to a CSV file
transformed_df.to_csv(output_file_path, index=False)